In [1]:
from intake_engine.app_flow import IntakeAppFlow

app = IntakeAppFlow(conn = None)
print(app.llm_config)
print(type(app.adapter).__name__)

{'backend': 'ollama', 'gguf_model_path': 'gemma-2-9b-it-Q4_K_M.gguf', 'ollama_model_name': 'gemma3:4b', 'ollama_base_url': 'http://localhost:11434', 'n_ctx': 2048, 'n_gpu_layers': -1, 'max_tokens': 200, 'temperature': 0.1}
OllamaAdapter


In [2]:
from intake_engine.app_flow import IntakeAppFlow
from pprint import pprint

app = IntakeAppFlow(conn = None)

app.new_session(
    session_name = "integration smoke test",
    primary_complaint = "headache",
    auto_save = False,
)

step = app.start_intake(auto_save = False)
pprint(step)

{'intent': 'ask_main_reason_for_visit',
 'question': 'Could you tell me what’s been bothering you today?',
 'target': None}


In [3]:
from pprint import pprint
from intake_engine.app_flow import IntakeAppFlow

app = IntakeAppFlow(conn = None)

app.new_session(
    session_name = "headache smoke test",
    primary_complaint = "headache",
    auto_save = False,
)

answers = [
    "I have had a headache since this morning.",
    "No, it did not start suddenly and severely.",
    "No weakness, numbness, trouble speaking, or similar symptoms.",
    "No vision changes.",
    "No confusion.",
    "No fever or neck stiffness.",
    "No recent head trauma.",
    "No, I am not pregnant and not postpartum.",
    "This morning.",
    "About four hours.",
    "7 out of 10.",
    "It comes and goes.",
    "It has stayed about the same.",
    "In the front of my head.",
    "Throbbing.",
]

pprint(app.start_intake(auto_save = False))

for answer in answers:
    result = app.submit_answer(answer, auto_save = False)
    pprint(result)

state = app.get_state()
print("\nFINAL HPI")
pprint(state["hpi"])

print("\nMISSING CLARIFICATIONS")
pprint(state["missing_clarifications"])

{'intent': 'ask_main_reason_for_visit',
 'question': 'Could you tell me what’s been bothering you today?',
 'target': None}
{'applied_update': {'append_fields': {},
                    'flags_to_add': [],
                    'missing_clarifications_to_add': [],
                    'set_fields': {'chief_complaint_primary': 'headache',
                                   'conversation_meta.early_exit_reason': None,
                                   'conversation_meta.last_answer_status': 'answered'}},
 'current_intent': 'ask_main_reason_for_visit',
 'missing_clarifications': ['when it started',
                            'where the symptom is located',
                            'how long it has been going on',
                            'how severe it is',
                            'associated symptoms'],
 'next_intent': 'ask_sudden_severe_onset',
 'next_question': 'Did the headache start suddenly and become very severe '
                  'right away?',
 'next_target': 'sudden_sev

In [4]:
from intake_engine.policies.policy_selector import get_policy_for_complaint

policy = get_policy_for_complaint("headache")
print(policy.policy_name)
print(policy.critical_followups)
print(policy.must_characterize)
print(policy.wrap_up_rule)

headache
['sudden_severe_onset', 'neurologic_symptoms', 'visual_changes', 'confusion_or_ams', 'fever_or_neck_stiffness', 'head_trauma', 'pregnancy_or_postpartum_context']
['onset', 'duration', 'severity', 'timing', 'course', 'location', 'character', 'aggravating_factors', 'relieving_factors', 'associated_symptoms']
{'type': 'characterization_threshold', 'require_all_critical': True, 'required_characterization_targets': ['onset', 'duration', 'severity', 'location', 'character', 'associated_symptoms'], 'min_required_characterization_count': 4}


In [5]:
from pprint import pprint

print("CURRENT STATE BEFORE CONTINUATION")
pprint(app.get_state()["hpi"])
print()
print("MISSING CLARIFICATIONS")
pprint(app.get_state()["missing_clarifications"])
print("-" * 80)

continuation_answers = [
    "Nausea and sensitivity to light.",
    "Bright light makes it worse.",
    "Rest helps a little.",
]

for i, answer in enumerate(continuation_answers, start = 1):
    print(f"CONTINUATION STEP {i}")
    print("PATIENT ANSWER:", answer)
    print()

    result = app.submit_answer(
        patient_answer = answer,
        auto_save = False,
    )

    pprint(result)
    print()

    state = app.get_state()

    print("CURRENT HPI")
    pprint(state["hpi"])
    print()

    print("CURRENT POLICY ANSWERS")
    pprint(state["policy_answers"])
    print()

    print("MISSING CLARIFICATIONS")
    pprint(state["missing_clarifications"])
    print("-" * 80)

    if not state["missing_clarifications"]:
        print("All missing clarifications resolved.")
        break

CURRENT STATE BEFORE CONTINUATION
{'aggravating_factors': [],
 'associated_symptoms': [],
 'character': 'Throbbing.',
 'context': None,
 'course': 'It has stayed about the same.',
 'duration': 'about four hours.',
 'location': 'In the front of my head.',
 'onset': 'This morning.',
 'relieving_factors': [],
 'severity': '7/10',
 'summary': None,
 'timing': 'It comes and goes.'}

MISSING CLARIFICATIONS
['associated symptoms']
--------------------------------------------------------------------------------
CONTINUATION STEP 1
PATIENT ANSWER: Nausea and sensitivity to light.

EXTRACTOR FALLBACK: ValueError append_fields must be a dictionary
{'applied_update': {'append_fields': {'hpi.aggravating_factors': ['nausea',
                                                                  'sensitivity '
                                                                  'to light.']},
                    'flags_to_add': [],
                    'missing_clarifications_to_add': [],
                    

In [6]:
from pprint import pprint

result = app.submit_answer(
    patient_answer = "Nausea and sensitivity to light.",
    auto_save = False,
)

pprint(result)
print()
pprint(app.get_state()["hpi"])
print()
pprint(app.get_state()["policy_answers"])
print()
pprint(app.get_state()["missing_clarifications"])

EXTRACTOR FALLBACK: ValueError append_fields must be a dictionary
{'applied_update': {'append_fields': {'hpi.aggravating_factors': ['nausea',
                                                                  'sensitivity '
                                                                  'to light.']},
                    'flags_to_add': [],
                    'missing_clarifications_to_add': [],
                    'set_fields': {'conversation_meta.early_exit_reason': None,
                                   'conversation_meta.last_answer_status': 'answered'}},
 'current_intent': 'ask_aggravating_factors',
 'missing_clarifications': ['associated symptoms'],
 'next_intent': 'ask_aggravating_factors',
 'next_question': 'Can you tell me what makes your headache worse?',
 'next_target': 'aggravating_factors'}

{'aggravating_factors': ['nausea',
                         'sensitivity to light.',
                         'bright light makes it worse.',
                         'rest helps a

In [7]:
from pprint import pprint
from intake_engine.app_flow import IntakeAppFlow

app = IntakeAppFlow(conn = None)

app.new_session(
    session_name = "abdominal pain smoke test",
    primary_complaint = "abdominal pain",
    auto_save = False,
)

print(app.start_intake(auto_save = False))

result = app.submit_answer("I have abdominal pain.", auto_save = False)
pprint(result)

result = app.submit_answer("No vomiting", auto_save = False)
pprint(result)

print()
pprint(app.get_state()["policy_answers"])

{'intent': 'ask_main_reason_for_visit', 'target': None, 'question': 'Could you tell me what’s been bothering you today?'}
{'applied_update': {'append_fields': {},
                    'flags_to_add': [],
                    'missing_clarifications_to_add': [],
                    'set_fields': {'chief_complaint_primary': 'abdominal pain',
                                   'conversation_meta.early_exit_reason': None,
                                   'conversation_meta.last_answer_status': 'answered'}},
 'current_intent': 'ask_main_reason_for_visit',
 'missing_clarifications': ['when it started',
                            'where the symptom is located',
                            'how long it has been going on',
                            'how severe it is',
                            'associated symptoms'],
 'next_intent': 'ask_onset',
 'next_question': 'When did your abdominal pain start?',
 'next_target': 'onset'}
{'applied_update': {'append_fields': {},
                    'fl

In [8]:
from intake_engine.policies.policy_selector import get_policy_for_complaint

policy = get_policy_for_complaint("abdominal pain")

print(type(policy).__name__)
print(policy.policy_name)
print(policy.critical_followups)
print(policy.must_characterize)
print(policy.high_priority_followups)
print(policy.choose_next_target({
    "chief_complaint_primary": "abdominal pain",
    "hpi": {
        "summary": None,
        "onset": None,
        "location": None,
        "duration": None,
        "timing": None,
        "course": None,
        "character": None,
        "severity": None,
        "aggravating_factors": [],
        "relieving_factors": [],
        "associated_symptoms": [],
        "context": None,
    },
    "policy_answers": {
        "vomiting": None,
        "fever": None,
        "bloody_stool_or_melena": None,
        "pregnancy_or_postpartum_context": None,
        "syncope_or_presyncope": None,
    },
}))

BaseComplaintPolicy
generic
[]
['onset', 'location', 'duration', 'severity', 'timing', 'course', 'character', 'aggravating_factors', 'relieving_factors', 'associated_symptoms']
['medications', 'allergies']
onset


In [9]:
import importlib
import intake_engine.policies.policy_selector

importlib.reload(intake_engine.policies.policy_selector)

from intake_engine.policies.policy_selector import get_policy_for_complaint

policy = get_policy_for_complaint("abdominal pain")

print(type(policy).__name__)
print(policy.policy_name)
print(policy.critical_followups)
print(policy.must_characterize)
print(policy.high_priority_followups)

BaseComplaintPolicy
generic
[]
['onset', 'location', 'duration', 'severity', 'timing', 'course', 'character', 'aggravating_factors', 'relieving_factors', 'associated_symptoms']
['medications', 'allergies']


In [10]:
import importlib
import intake_engine.policies.complaint_policies
import intake_engine.policies.policy_selector

importlib.reload(intake_engine.policies.complaint_policies)
importlib.reload(intake_engine.policies.policy_selector)

<module 'intake_engine.policies.policy_selector' from 'c:\\Source\\689\\Capstone2026_Spring\\intake_engine\\policies\\policy_selector.py'>

In [11]:
import importlib
import intake_engine.policies.complaint_policies
import intake_engine.policies.policy_selector

importlib.reload(intake_engine.policies.complaint_policies)
importlib.reload(intake_engine.policies.policy_selector)

<module 'intake_engine.policies.policy_selector' from 'c:\\Source\\689\\Capstone2026_Spring\\intake_engine\\policies\\policy_selector.py'>

In [12]:
from intake_engine.policies.policy_selector import get_policy_for_complaint

policy = get_policy_for_complaint("abdominal pain")

print(type(policy).__name__)
print(policy.policy_name)
print(policy.critical_followups)
print(policy.must_characterize)
print(policy.high_priority_followups)

BaseComplaintPolicy
abdominal_pain
['vomiting', 'fever', 'bloody_stool_or_melena', 'pregnancy_or_postpartum_context', 'syncope_or_presyncope']
['onset', 'duration', 'severity', 'timing', 'course', 'location', 'character', 'aggravating_factors', 'relieving_factors', 'associated_symptoms']
['medications', 'allergies', 'nausea', 'diarrhea', 'constipation', 'urinary_symptoms', 'last_oral_intake']


In [13]:
from pprint import pprint
from intake_engine.app_flow import IntakeAppFlow

app = IntakeAppFlow(conn = None)

app.new_session(
    session_name = "abdominal pain smoke test",
    primary_complaint = "abdominal pain",
    auto_save = False,
)

print("INITIAL QUESTION")
pprint(app.start_intake(auto_save = False))
print()

result = app.submit_answer("I have abdominal pain.", auto_save = False)
print("AFTER CHIEF COMPLAINT")
pprint(result)
print()

result = app.submit_answer("No vomiting", auto_save = False)
print("AFTER NO VOMITING")
pprint(result)
print()

print("CURRENT POLICY ANSWERS")
pprint(app.get_state()["policy_answers"])

INITIAL QUESTION
{'intent': 'ask_main_reason_for_visit',
 'question': 'Could you tell me what’s been bothering you today?',
 'target': None}

AFTER CHIEF COMPLAINT
{'applied_update': {'append_fields': {},
                    'flags_to_add': [],
                    'missing_clarifications_to_add': [],
                    'set_fields': {'chief_complaint_primary': 'abdominal pain',
                                   'conversation_meta.early_exit_reason': None,
                                   'conversation_meta.last_answer_status': 'answered'}},
 'current_intent': 'ask_main_reason_for_visit',
 'missing_clarifications': ['when it started',
                            'where the symptom is located',
                            'how long it has been going on',
                            'how severe it is',
                            'associated symptoms'],
 'next_intent': 'ask_vomiting',
 'next_question': 'Are you experiencing any vomiting?',
 'next_target': 'vomiting'}

AFTER NO VOMITIN

In [14]:
import importlib
import intake_engine.state.rule_based_parser
import intake_engine.IntakeState
import intake_engine.app_flow

importlib.reload(intake_engine.state.rule_based_parser)
importlib.reload(intake_engine.IntakeState)
importlib.reload(intake_engine.app_flow)

<module 'intake_engine.app_flow' from 'c:\\Source\\689\\Capstone2026_Spring\\intake_engine\\app_flow.py'>

In [15]:
from pprint import pprint
from intake_engine.app_flow import IntakeAppFlow

app = IntakeAppFlow(conn = None)

app.new_session(
    session_name = "abdominal pain smoke test",
    primary_complaint = "abdominal pain",
    auto_save = False,
)

print("INITIAL QUESTION")
pprint(app.start_intake(auto_save = False))
print()

result = app.submit_answer("I have abdominal pain.", auto_save = False)
print("AFTER CHIEF COMPLAINT")
pprint(result)
print()

result = app.submit_answer("No vomiting", auto_save = False)
print("AFTER NO VOMITING")
pprint(result)
print()

print("CURRENT POLICY ANSWERS")
pprint(app.get_state()["policy_answers"])

INITIAL QUESTION
{'intent': 'ask_main_reason_for_visit',
 'question': 'Could you tell me what’s been bothering you today?',
 'target': None}

AFTER CHIEF COMPLAINT
{'applied_update': {'append_fields': {},
                    'flags_to_add': [],
                    'missing_clarifications_to_add': [],
                    'set_fields': {'chief_complaint_primary': 'abdominal pain',
                                   'conversation_meta.early_exit_reason': None,
                                   'conversation_meta.last_answer_status': 'answered'}},
 'current_intent': 'ask_main_reason_for_visit',
 'missing_clarifications': ['when it started',
                            'where the symptom is located',
                            'how long it has been going on',
                            'how severe it is',
                            'associated symptoms'],
 'next_intent': 'ask_vomiting',
 'next_question': 'Are you currently experiencing any vomiting?',
 'next_target': 'vomiting'}

AFTER 

In [16]:
import importlib
import intake_engine.IntakeState
import intake_engine.app_flow

importlib.reload(intake_engine.IntakeState)
importlib.reload(intake_engine.app_flow)

<module 'intake_engine.app_flow' from 'c:\\Source\\689\\Capstone2026_Spring\\intake_engine\\app_flow.py'>

In [17]:
from pprint import pprint
from intake_engine.app_flow import IntakeAppFlow

app = IntakeAppFlow(conn = None)

app.new_session(
    session_name = "abdominal pain smoke test",
    primary_complaint = "abdominal pain",
    auto_save = False,
)

print("INITIAL QUESTION")
pprint(app.start_intake(auto_save = False))
print()

result = app.submit_answer("I have abdominal pain.", auto_save = False)
print("AFTER CHIEF COMPLAINT")
pprint(result)
print()

result = app.submit_answer("No vomiting", auto_save = False)
print("AFTER NO VOMITING")
pprint(result)
print()

print("CURRENT POLICY ANSWERS")
pprint(app.get_state()["policy_answers"])

INITIAL QUESTION
{'intent': 'ask_main_reason_for_visit',
 'question': 'Could you tell me what’s been bothering you today?',
 'target': None}

AFTER CHIEF COMPLAINT
{'applied_update': {'append_fields': {},
                    'flags_to_add': [],
                    'missing_clarifications_to_add': [],
                    'set_fields': {'chief_complaint_primary': 'abdominal pain',
                                   'conversation_meta.early_exit_reason': None,
                                   'conversation_meta.last_answer_status': 'answered'}},
 'current_intent': 'ask_main_reason_for_visit',
 'missing_clarifications': ['when it started',
                            'where the symptom is located',
                            'how long it has been going on',
                            'how severe it is',
                            'associated symptoms'],
 'next_intent': 'ask_vomiting',
 'next_question': 'Are you currently experiencing any vomiting?',
 'next_target': 'vomiting'}

AFTER 

In [19]:
import importlib

import intake_engine.policies.target_specs
import intake_engine.state.rule_based_parser
import intake_engine.IntakeState
import intake_engine.app_flow

importlib.reload(intake_engine.policies.target_specs)
importlib.reload(intake_engine.state.rule_based_parser)
importlib.reload(intake_engine.IntakeState)
importlib.reload(intake_engine.app_flow)

<module 'intake_engine.app_flow' from 'c:\\Source\\689\\Capstone2026_Spring\\intake_engine\\app_flow.py'>

In [20]:
from intake_engine.IntakeState import IntakeState
from intake_engine.state.rule_based_parser import extract_yes_no_unknown
from intake_engine.policies.target_specs import TARGET_SPECS

state = IntakeState()

intent = "ask_vomiting"
patient_answer = "No vomiting"

print("YES/NO PARSE:", extract_yes_no_unknown(patient_answer))
print("TARGET:", state._intent_to_target(intent))
print("SPEC:", TARGET_SPECS[state._intent_to_target(intent)])

test_update = {
    "set_fields": {
        "conversation_meta.last_answer_status": "answered",
        "conversation_meta.early_exit_reason": None,
    },
    "append_fields": {},
    "flags_to_add": [],
    "missing_clarifications_to_add": [],
}

result = state._postprocess_llm_update_for_intent(
    intent = intent,
    patient_answer = patient_answer,
    applied_update = test_update,
)

print("POSTPROCESSED UPDATE:")
print(result)

YES/NO PARSE: False
TARGET: vomiting
SPEC: {'intent': 'ask_vomiting', 'state_path': 'policy_answers.vomiting', 'fallback_parse_mode': 'yes_no', 'question_mode': 'deterministic', 'question_text': 'Have you been vomiting?', 'question_instruction': 'Ask whether the patient has been vomiting.', 'extraction_instruction': 'Extract a boolean into set_fields under "policy_answers.vomiting".'}
POSTPROCESSED UPDATE:
{'set_fields': {'conversation_meta.last_answer_status': 'answered', 'conversation_meta.early_exit_reason': None, 'policy_answers.vomiting': False}, 'append_fields': {}, 'flags_to_add': [], 'missing_clarifications_to_add': []}


In [21]:
from pprint import pprint
from intake_engine.app_flow import IntakeAppFlow

app = IntakeAppFlow(conn = None)

app.new_session(
    session_name = "abdominal pain smoke test",
    primary_complaint = "abdominal pain",
    auto_save = False,
)

print("INITIAL QUESTION")
pprint(app.start_intake(auto_save = False))
print()

result = app.submit_answer("I have abdominal pain.", auto_save = False)
print("AFTER CHIEF COMPLAINT")
pprint(result)
print()

result = app.submit_answer("No vomiting", auto_save = False)
print("AFTER NO VOMITING")
pprint(result)
print()

print("CURRENT POLICY ANSWERS")
pprint(app.get_state()["policy_answers"])

INITIAL QUESTION
{'intent': 'ask_main_reason_for_visit',
 'question': 'Could you tell me what’s been bothering you today?',
 'target': None}

AFTER CHIEF COMPLAINT
{'applied_update': {'append_fields': {},
                    'flags_to_add': [],
                    'missing_clarifications_to_add': [],
                    'set_fields': {'chief_complaint_primary': 'abdominal pain',
                                   'conversation_meta.early_exit_reason': None,
                                   'conversation_meta.last_answer_status': 'answered'}},
 'current_intent': 'ask_main_reason_for_visit',
 'missing_clarifications': ['when it started',
                            'where the symptom is located',
                            'how long it has been going on',
                            'how severe it is',
                            'associated symptoms'],
 'next_intent': 'ask_vomiting',
 'next_question': 'Are you currently experiencing any vomiting?',
 'next_target': 'vomiting'}

AFTER 

In [22]:
from intake_engine.policies.policy_selector import get_policy_for_complaint

policy = get_policy_for_complaint("abdominal pain")

fake_intake = {
    "chief_complaint_primary": "abdominal pain",
    "hpi": {
        "summary": None,
        "onset": None,
        "location": None,
        "duration": None,
        "timing": None,
        "course": None,
        "character": None,
        "severity": None,
        "aggravating_factors": [],
        "relieving_factors": [],
        "associated_symptoms": [],
        "context": None,
    },
    "policy_answers": {
        "vomiting": False,
        "fever": None,
        "bloody_stool_or_melena": None,
        "pregnancy_or_postpartum_context": None,
        "syncope_or_presyncope": None,
    },
}

print("policy_name =", policy.policy_name)
print("critical_followups =", policy.critical_followups)
print("vomiting resolved =", policy._is_target_resolved(fake_intake, "vomiting"))
print("next target =", policy.choose_next_target(fake_intake))

policy_name = abdominal_pain
critical_followups = ['vomiting', 'fever', 'bloody_stool_or_melena', 'pregnancy_or_postpartum_context', 'syncope_or_presyncope']
vomiting resolved = False
next target = vomiting


In [23]:
import importlib
import intake_engine.policies.base_complaint_policy
import intake_engine.policies.policy_selector
import intake_engine.IntakeState
import intake_engine.app_flow

importlib.reload(intake_engine.policies.base_complaint_policy)
importlib.reload(intake_engine.policies.policy_selector)
importlib.reload(intake_engine.IntakeState)
importlib.reload(intake_engine.app_flow)

<module 'intake_engine.app_flow' from 'c:\\Source\\689\\Capstone2026_Spring\\intake_engine\\app_flow.py'>

In [24]:
from pprint import pprint
from intake_engine.app_flow import IntakeAppFlow

app = IntakeAppFlow(conn = None)

app.new_session(
    session_name = "abdominal pain smoke test",
    primary_complaint = "abdominal pain",
    auto_save = False,
)

print("INITIAL QUESTION")
pprint(app.start_intake(auto_save = False))
print()

result = app.submit_answer("I have abdominal pain.", auto_save = False)
print("AFTER CHIEF COMPLAINT")
pprint(result)
print()

result = app.submit_answer("No vomiting", auto_save = False)
print("AFTER NO VOMITING")
pprint(result)
print()

print("CURRENT POLICY ANSWERS")
pprint(app.get_state()["policy_answers"])

INITIAL QUESTION
{'intent': 'ask_main_reason_for_visit',
 'question': 'Could you tell me what’s been bothering you today?',
 'target': None}

AFTER CHIEF COMPLAINT
{'applied_update': {'append_fields': {},
                    'flags_to_add': [],
                    'missing_clarifications_to_add': [],
                    'set_fields': {'chief_complaint_primary': 'abdominal pain',
                                   'conversation_meta.early_exit_reason': None,
                                   'conversation_meta.last_answer_status': 'answered'}},
 'current_intent': 'ask_main_reason_for_visit',
 'missing_clarifications': ['when it started',
                            'where the symptom is located',
                            'how long it has been going on',
                            'how severe it is',
                            'associated symptoms'],
 'next_intent': 'ask_vomiting',
 'next_question': 'Are you currently experiencing any vomiting?',
 'next_target': 'vomiting'}

AFTER 

In [2]:
import importlib
import intake_engine.policies.target_specs
import intake_engine.state.templates
import intake_engine.state.rule_based_parser
import intake_engine.IntakeState
import intake_engine.app_flow

importlib.reload(intake_engine.policies.target_specs)
importlib.reload(intake_engine.state.templates)
importlib.reload(intake_engine.state.rule_based_parser)
importlib.reload(intake_engine.IntakeState)
importlib.reload(intake_engine.app_flow)

<module 'intake_engine.app_flow' from 'c:\\Source\\689\\Capstone2026_Spring\\intake_engine\\app_flow.py'>

In [3]:
from pprint import pprint
from intake_engine.app_flow import IntakeAppFlow

app = IntakeAppFlow(conn = None)

app.new_session(
    session_name = "dizziness smoke test",
    primary_complaint = "dizziness",
    auto_save = False,
)

answers = [
    "I have dizziness.",
    "No, I have not fainted or felt like I might faint.",
    "No weakness, numbness, trouble speaking, or similar symptoms.",
    "No chest pain.",
    "No shortness of breath.",
    "No recent head trauma.",
    "This morning.",
    "About three hours.",
    "6 out of 10.",
    "It comes and goes.",
    "It has stayed about the same.",
    "Standing up makes it worse.",
    "Sitting down helps.",
    "Nausea.",
]

print("INITIAL QUESTION")
pprint(app.start_intake(auto_save = False))
print()

for i, answer in enumerate(answers, start = 1):
    print(f"STEP {i}")
    print("PATIENT ANSWER:", answer)
    print()

    result = app.submit_answer(
        patient_answer = answer,
        auto_save = False,
    )

    pprint(result)
    print()

    print("CURRENT HPI")
    pprint(app.get_state()["hpi"])
    print()

    print("CURRENT POLICY ANSWERS")
    pprint(app.get_state()["policy_answers"])
    print()

    print("MISSING CLARIFICATIONS")
    pprint(app.get_state()["missing_clarifications"])
    print("-" * 80)

INITIAL QUESTION
{'intent': 'ask_main_reason_for_visit',
 'question': 'Could you tell me what’s been bothering you today?',
 'target': None}

STEP 1
PATIENT ANSWER: I have dizziness.

{'applied_update': {'append_fields': {},
                    'flags_to_add': [],
                    'missing_clarifications_to_add': [],
                    'set_fields': {'chief_complaint_primary': 'dizziness',
                                   'conversation_meta.early_exit_reason': None,
                                   'conversation_meta.last_answer_status': 'answered'}},
 'current_intent': 'ask_main_reason_for_visit',
 'missing_clarifications': ['when it started',
                            'where the symptom is located',
                            'how long it has been going on',
                            'how severe it is',
                            'associated symptoms'],
 'next_intent': 'ask_syncope_or_presyncope',
 'next_question': 'Have you fainted or felt like you might faint?',
 'nex

In [7]:
from intake_engine.policies.policy_selector import get_policy_for_complaint

policy = get_policy_for_complaint("dizziness")

fake_intake = {
    "chief_complaint_primary": "dizziness",
    "hpi": {
        "summary": None,
        "onset": None,
        "location": None,
        "duration": None,
        "timing": None,
        "course": None,
        "character": None,
        "severity": None,
        "aggravating_factors": [],
        "relieving_factors": [],
        "associated_symptoms": [],
        "context": None,
    },
    "policy_answers": {
        "syncope_or_presyncope": False,
        "neurologic_symptoms": False,
        "chest_pain": False,
        "shortness_of_breath": None,
        "head_trauma": None,
    },
}

print("chest_pain resolved =", policy._is_target_resolved(fake_intake, "chest_pain"))
print("next target =", policy.choose_next_target(fake_intake))

chest_pain resolved = True
next target = shortness_of_breath


In [1]:
import importlib
import intake_engine.policies.target_specs
import intake_engine.policies.base_complaint_policy
import intake_engine.policies.complaint_policies
import intake_engine.policies.policy_selector
import intake_engine.IntakeState
import intake_engine.app_flow

importlib.reload(intake_engine.policies.target_specs)
importlib.reload(intake_engine.policies.base_complaint_policy)
importlib.reload(intake_engine.policies.complaint_policies)
importlib.reload(intake_engine.policies.policy_selector)
importlib.reload(intake_engine.IntakeState)
importlib.reload(intake_engine.app_flow)

<module 'intake_engine.app_flow' from 'c:\\Source\\689\\Capstone2026_Spring\\intake_engine\\app_flow.py'>

In [2]:
from intake_engine.policies.policy_selector import get_policy_for_complaint

policy = get_policy_for_complaint("dizziness")

fake_intake = {
    "chief_complaint_primary": "dizziness",
    "hpi": {
        "summary": None,
        "onset": None,
        "location": None,
        "duration": None,
        "timing": None,
        "course": None,
        "character": None,
        "severity": None,
        "aggravating_factors": [],
        "relieving_factors": [],
        "associated_symptoms": [],
        "context": None,
    },
    "policy_answers": {
        "syncope_or_presyncope": False,
        "neurologic_symptoms": False,
        "chest_pain": False,
        "shortness_of_breath": None,
        "head_trauma": None,
    },
}

print("chest_pain resolved =", policy._is_target_resolved(fake_intake, "chest_pain"))
print("next target =", policy.choose_next_target(fake_intake))

chest_pain resolved = True
next target = shortness_of_breath


In [1]:
import importlib
import intake_engine.policies.target_specs
import intake_engine.policies.base_complaint_policy
import intake_engine.policies.complaint_policies
import intake_engine.policies.policy_selector
import intake_engine.state.templates
import intake_engine.state.routing
import intake_engine.state.rule_based_parser
import intake_engine.llm_extractor
import intake_engine.IntakeState
import intake_engine.app_flow

importlib.reload(intake_engine.policies.target_specs)
importlib.reload(intake_engine.policies.base_complaint_policy)
importlib.reload(intake_engine.policies.complaint_policies)
importlib.reload(intake_engine.policies.policy_selector)
importlib.reload(intake_engine.state.templates)
importlib.reload(intake_engine.state.routing)
importlib.reload(intake_engine.state.rule_based_parser)
importlib.reload(intake_engine.llm_extractor)
importlib.reload(intake_engine.IntakeState)
importlib.reload(intake_engine.app_flow)

<module 'intake_engine.app_flow' from 'c:\\Source\\689\\Capstone2026_Spring\\intake_engine\\app_flow.py'>

In [2]:
from pprint import pprint
from intake_engine.app_flow import IntakeAppFlow

app = IntakeAppFlow(conn = None)

app.new_session(
    session_name = "dizziness fresh test",
    primary_complaint = "dizziness",
    auto_save = False,
)

print("INITIAL QUESTION")
pprint(app.start_intake(auto_save = False))
print()

answers = [
    "I have dizziness.",
    "No, I have not fainted or felt like I might faint.",
    "No weakness, numbness, trouble speaking, or similar symptoms.",
    "No chest pain.",
    "No shortness of breath.",
]

for i, answer in enumerate(answers, start = 1):
    print(f"STEP {i}")
    print("PATIENT ANSWER:", answer)
    result = app.submit_answer(answer, auto_save = False)
    pprint(result)
    print()
    print("CURRENT POLICY ANSWERS")
    pprint(app.get_state()["policy_answers"])
    print("-" * 80)

INITIAL QUESTION
{'intent': 'ask_main_reason_for_visit',
 'question': 'Could you tell me what’s been bothering you today?',
 'target': None}

STEP 1
PATIENT ANSWER: I have dizziness.
{'applied_update': {'append_fields': {},
                    'flags_to_add': [],
                    'missing_clarifications_to_add': [],
                    'set_fields': {'chief_complaint_primary': 'dizziness',
                                   'conversation_meta.early_exit_reason': None,
                                   'conversation_meta.last_answer_status': 'answered'}},
 'current_intent': 'ask_main_reason_for_visit',
 'missing_clarifications': ['when it started',
                            'where the symptom is located',
                            'how long it has been going on',
                            'how severe it is',
                            'associated symptoms'],
 'next_intent': 'ask_syncope_or_presyncope',
 'next_question': 'Have you fainted or felt like you might faint?',
 'next

In [2]:
import importlib

import intake_engine.policies.target_specs
import intake_engine.policies.base_complaint_policy
import intake_engine.policies.complaint_policies
import intake_engine.policies.policy_selector
import intake_engine.state.templates
import intake_engine.state.routing
import intake_engine.state.rule_based_parser
import intake_engine.llm_extractor
import intake_engine.IntakeState
import intake_engine.app_flow

importlib.reload(intake_engine.policies.target_specs)
importlib.reload(intake_engine.policies.base_complaint_policy)
importlib.reload(intake_engine.policies.complaint_policies)
importlib.reload(intake_engine.policies.policy_selector)
importlib.reload(intake_engine.state.templates)
importlib.reload(intake_engine.state.routing)
importlib.reload(intake_engine.state.rule_based_parser)
importlib.reload(intake_engine.llm_extractor)
importlib.reload(intake_engine.IntakeState)
importlib.reload(intake_engine.app_flow)

<module 'intake_engine.app_flow' from 'c:\\Source\\689\\Capstone2026_Spring\\intake_engine\\app_flow.py'>

In [3]:
from pprint import pprint
from intake_engine.app_flow import IntakeAppFlow

app = IntakeAppFlow(conn = None)

app.new_session(
    session_name = "headache list-answer regression test",
    primary_complaint = "headache",
    auto_save = False,
)

answers = [
    "I have a headache.",
    "No.",
    "No weakness, numbness, trouble speaking, or similar symptoms.",
    "No vision changes.",
    "No confusion.",
    "No fever or neck stiffness.",
    "No recent head trauma.",
    "No, I am not pregnant and not postpartum.",
    "This morning.",
    "About four hours.",
    "7 out of 10.",
    "It comes and goes.",
    "It has stayed about the same.",
    "In the front of my head.",
    "Throbbing.",
]

print("INITIAL QUESTION")
pprint(app.start_intake(auto_save = False))
print()

for answer in answers:
    result = app.submit_answer(answer, auto_save = False)
    pprint(result)

print("\nSTATE BEFORE LIST TEST")
pprint(app.get_state()["hpi"])
print()
pprint(app.get_state()["conversation_meta"]["question_status"])
print()
pprint(app.get_state()["missing_clarifications"])

INITIAL QUESTION
{'intent': 'ask_main_reason_for_visit',
 'question': 'Could you tell me what’s been bothering you today?',
 'target': None}

{'applied_update': {'append_fields': {},
                    'flags_to_add': [],
                    'missing_clarifications_to_add': [],
                    'set_fields': {'chief_complaint_primary': 'headache',
                                   'conversation_meta.early_exit_reason': None,
                                   'conversation_meta.last_answer_status': 'answered'}},
 'current_intent': 'ask_main_reason_for_visit',
 'missing_clarifications': ['when it started',
                            'where the symptom is located',
                            'how long it has been going on',
                            'how severe it is',
                            'associated symptoms'],
 'next_intent': 'ask_sudden_severe_onset',
 'next_question': 'Did the headache start suddenly and become very severe '
                  'right away?',
 'next_ta

In [4]:
from pprint import pprint

result = app.submit_answer("No", auto_save = False)

print("AFTER 'No'")
pprint(result)
print()
print("QUESTION STATUS")
pprint(app.get_state()["conversation_meta"]["question_status"])
print()
print("HPI")
pprint(app.get_state()["hpi"])
print()
print("MISSING CLARIFICATIONS")
pprint(app.get_state()["missing_clarifications"])

AFTER 'No'
{'applied_update': {'append_fields': {},
                    'flags_to_add': [],
                    'missing_clarifications_to_add': [],
                    'set_fields': {'conversation_meta.early_exit_reason': None,
                                   'conversation_meta.last_answer_status': 'answered'}},
 'current_intent': 'ask_aggravating_factors',
 'missing_clarifications': ['associated symptoms'],
 'next_intent': 'ask_relieving_factors',
 'next_question': 'What makes the headache feel better?',
 'next_target': 'relieving_factors'}

QUESTION STATUS
{'aggravating_factors': 'asked_answered',
 'character': 'asked_answered',
 'confusion_or_ams': 'asked_answered',
 'course': 'asked_answered',
 'duration': 'asked_answered',
 'fever_or_neck_stiffness': 'asked_answered',
 'head_trauma': 'asked_answered',
 'location': 'asked_answered',
 'neurologic_symptoms': 'asked_answered',
 'onset': 'asked_answered',
 'pregnancy_or_postpartum_context': 'asked_answered',
 'severity': 'asked_ans

In [5]:
from pprint import pprint
from intake_engine.app_flow import IntakeAppFlow

app = IntakeAppFlow(conn = None)

app.new_session(
    session_name = "headache decline regression test",
    primary_complaint = "headache",
    auto_save = False,
)

answers = [
    "I have a headache.",
    "No.",
    "No weakness, numbness, trouble speaking, or similar symptoms.",
    "No vision changes.",
    "No confusion.",
    "No fever or neck stiffness.",
    "No recent head trauma.",
    "No, I am not pregnant and not postpartum.",
    "This morning.",
    "About four hours.",
    "7 out of 10.",
    "It comes and goes.",
    "It has stayed about the same.",
    "In the front of my head.",
    "Throbbing.",
]

pprint(app.start_intake(auto_save = False))

for answer in answers:
    app.submit_answer(answer, auto_save = False)

result = app.submit_answer("Skip", auto_save = False)

print("AFTER 'Skip'")
pprint(result)
print()
print("QUESTION STATUS")
pprint(app.get_state()["conversation_meta"]["question_status"])
print()
print("HPI")
pprint(app.get_state()["hpi"])

{'intent': 'ask_main_reason_for_visit',
 'question': 'Could you tell me what’s been bothering you today?',
 'target': None}
AFTER 'Skip'
{'applied_update': {'append_fields': {},
                    'flags_to_add': [],
                    'missing_clarifications_to_add': [],
                    'set_fields': {'conversation_meta.last_answer_status': 'declined'}},
 'current_intent': 'ask_aggravating_factors',
 'missing_clarifications': ['associated symptoms'],
 'next_intent': 'ask_relieving_factors',
 'next_question': 'What makes the headache feel better?',
 'next_target': 'relieving_factors'}

QUESTION STATUS
{'aggravating_factors': 'asked_declined',
 'character': 'asked_answered',
 'confusion_or_ams': 'asked_answered',
 'course': 'asked_answered',
 'duration': 'asked_answered',
 'fever_or_neck_stiffness': 'asked_answered',
 'head_trauma': 'asked_answered',
 'location': 'asked_answered',
 'neurologic_symptoms': 'asked_answered',
 'onset': 'asked_answered',
 'pregnancy_or_postpartum_conte

In [6]:
from pprint import pprint
from intake_engine.app_flow import IntakeAppFlow

app = IntakeAppFlow(conn = None)

app.new_session(
    session_name = "dizziness quick regression",
    primary_complaint = "dizziness",
    auto_save = False,
)

answers = [
    "I have dizziness.",
    "No, I have not fainted or felt like I might faint.",
    "No weakness, numbness, trouble speaking, or similar symptoms.",
    "No chest pain.",
    "No shortness of breath.",
]

print("INITIAL QUESTION")
pprint(app.start_intake(auto_save = False))
print()

for answer in answers:
    result = app.submit_answer(answer, auto_save = False)
    pprint(result)
    print()

print("CURRENT POLICY ANSWERS")
pprint(app.get_state()["policy_answers"])

INITIAL QUESTION
{'intent': 'ask_main_reason_for_visit',
 'question': 'Could you tell me what’s been bothering you today?',
 'target': None}

{'applied_update': {'append_fields': {},
                    'flags_to_add': [],
                    'missing_clarifications_to_add': [],
                    'set_fields': {'chief_complaint_primary': 'dizziness',
                                   'conversation_meta.early_exit_reason': None,
                                   'conversation_meta.last_answer_status': 'answered'}},
 'current_intent': 'ask_main_reason_for_visit',
 'missing_clarifications': ['when it started',
                            'where the symptom is located',
                            'how long it has been going on',
                            'how severe it is',
                            'associated symptoms'],
 'next_intent': 'ask_syncope_or_presyncope',
 'next_question': 'Have you fainted or felt like you might faint?',
 'next_target': 'syncope_or_presyncope'}

{'app

In [7]:
from pprint import pprint
from intake_engine.app_flow import IntakeAppFlow

app = IntakeAppFlow(conn = None)

app.new_session(
    session_name = "headache decline regression test",
    primary_complaint = "headache",
    auto_save = False,
)

answers = [
    "I have a headache.",
    "No.",
    "No weakness, numbness, trouble speaking, or similar symptoms.",
    "No vision changes.",
    "No confusion.",
    "No fever or neck stiffness.",
    "No recent head trauma.",
    "No, I am not pregnant and not postpartum.",
    "This morning.",
    "About four hours.",
    "7 out of 10.",
    "It comes and goes.",
    "It has stayed about the same.",
    "In the front of my head.",
    "Throbbing.",
]

pprint(app.start_intake(auto_save = False))

for answer in answers:
    app.submit_answer(answer, auto_save = False)

result = app.submit_answer("Skip", auto_save = False)

print("AFTER 'Skip'")
pprint(result)
print()
print("QUESTION STATUS")
pprint(app.get_state()["conversation_meta"]["question_status"])
print()
print("HPI")
pprint(app.get_state()["hpi"])

{'intent': 'ask_main_reason_for_visit',
 'question': 'Could you tell me what’s been bothering you today?',
 'target': None}
AFTER 'Skip'
{'applied_update': {'append_fields': {},
                    'flags_to_add': [],
                    'missing_clarifications_to_add': [],
                    'set_fields': {'conversation_meta.last_answer_status': 'declined'}},
 'current_intent': 'ask_aggravating_factors',
 'missing_clarifications': ['associated symptoms'],
 'next_intent': 'ask_relieving_factors',
 'next_question': 'What makes the headache feel better?',
 'next_target': 'relieving_factors'}

QUESTION STATUS
{'aggravating_factors': 'asked_declined',
 'character': 'asked_answered',
 'confusion_or_ams': 'asked_answered',
 'course': 'asked_answered',
 'duration': 'asked_answered',
 'fever_or_neck_stiffness': 'asked_answered',
 'head_trauma': 'asked_answered',
 'location': 'asked_answered',
 'neurologic_symptoms': 'asked_answered',
 'onset': 'asked_answered',
 'pregnancy_or_postpartum_conte

In [8]:
from pprint import pprint
from intake_engine.app_flow import IntakeAppFlow

app = IntakeAppFlow(conn = None)

app.new_session(
    session_name = "headache no regression test",
    primary_complaint = "headache",
    auto_save = False,
)

answers = [
    "I have a headache.",
    "No.",
    "No weakness, numbness, trouble speaking, or similar symptoms.",
    "No vision changes.",
    "No confusion.",
    "No fever or neck stiffness.",
    "No recent head trauma.",
    "No, I am not pregnant and not postpartum.",
    "This morning.",
    "About four hours.",
    "7 out of 10.",
    "It comes and goes.",
    "It has stayed about the same.",
    "In the front of my head.",
    "Throbbing.",
]

pprint(app.start_intake(auto_save = False))

for answer in answers:
    app.submit_answer(answer, auto_save = False)

result = app.submit_answer("No", auto_save = False)

print("AFTER 'No'")
pprint(result)
print()
print("QUESTION STATUS")
pprint(app.get_state()["conversation_meta"]["question_status"])
print()
print("HPI")
pprint(app.get_state()["hpi"])

{'intent': 'ask_main_reason_for_visit',
 'question': 'Could you tell me what’s been bothering you today?',
 'target': None}
AFTER 'No'
{'applied_update': {'append_fields': {},
                    'flags_to_add': [],
                    'missing_clarifications_to_add': [],
                    'set_fields': {'conversation_meta.early_exit_reason': None,
                                   'conversation_meta.last_answer_status': 'answered'}},
 'current_intent': 'ask_aggravating_factors',
 'missing_clarifications': ['associated symptoms'],
 'next_intent': 'ask_relieving_factors',
 'next_question': 'What makes the headache feel better?',
 'next_target': 'relieving_factors'}

QUESTION STATUS
{'aggravating_factors': 'asked_answered',
 'character': 'asked_answered',
 'confusion_or_ams': 'asked_answered',
 'course': 'asked_answered',
 'duration': 'asked_answered',
 'fever_or_neck_stiffness': 'asked_answered',
 'head_trauma': 'asked_answered',
 'location': 'asked_answered',
 'neurologic_symptoms':

In [9]:
import importlib

import intake_engine.policies.target_specs
import intake_engine.policies.base_complaint_policy
import intake_engine.policies.complaint_policies
import intake_engine.policies.policy_selector
import intake_engine.state.templates
import intake_engine.state.routing
import intake_engine.state.rule_based_parser
import intake_engine.llm_extractor
import intake_engine.IntakeState
import intake_engine.app_flow

importlib.reload(intake_engine.policies.target_specs)
importlib.reload(intake_engine.policies.base_complaint_policy)
importlib.reload(intake_engine.policies.complaint_policies)
importlib.reload(intake_engine.policies.policy_selector)
importlib.reload(intake_engine.state.templates)
importlib.reload(intake_engine.state.routing)
importlib.reload(intake_engine.state.rule_based_parser)
importlib.reload(intake_engine.llm_extractor)
importlib.reload(intake_engine.IntakeState)
importlib.reload(intake_engine.app_flow)

<module 'intake_engine.app_flow' from 'c:\\Source\\689\\Capstone2026_Spring\\intake_engine\\app_flow.py'>

In [10]:
from pprint import pprint
from intake_engine.app_flow import IntakeAppFlow

app = IntakeAppFlow(conn = None)

app.new_session(
    session_name = "shortness of breath smoke test",
    primary_complaint = "shortness of breath",
    auto_save = False,
)

answers = [
    "I have shortness of breath.",
    "No chest pain.",
    "No, it is not getting worse quickly.",
    "No fainting.",
    "No fever.",
    "No wheezing.",
    "This morning.",
    "About three hours.",
    "6 out of 10.",
    "It comes and goes.",
    "It has stayed about the same.",
    "Walking makes it worse.",
    "Rest helps.",
    "Cough.",
]

print("INITIAL QUESTION")
pprint(app.start_intake(auto_save = False))
print()

for i, answer in enumerate(answers, start = 1):
    print(f"STEP {i}")
    print("PATIENT ANSWER:", answer)
    print()

    result = app.submit_answer(
        patient_answer = answer,
        auto_save = False,
    )

    pprint(result)
    print()

    print("CURRENT HPI")
    pprint(app.get_state()["hpi"])
    print()

    print("CURRENT POLICY ANSWERS")
    pprint(app.get_state()["policy_answers"])
    print()

    print("MISSING CLARIFICATIONS")
    pprint(app.get_state()["missing_clarifications"])
    print("-" * 80)

INITIAL QUESTION
{'intent': 'ask_main_reason_for_visit',
 'question': 'Could you tell me what’s been bothering you today?',
 'target': None}

STEP 1
PATIENT ANSWER: I have shortness of breath.

{'applied_update': {'append_fields': {},
                    'flags_to_add': [],
                    'missing_clarifications_to_add': [],
                    'set_fields': {'chief_complaint_primary': 'shortness of '
                                                              'breath',
                                   'conversation_meta.early_exit_reason': None,
                                   'conversation_meta.last_answer_status': 'answered'}},
 'current_intent': 'ask_main_reason_for_visit',
 'missing_clarifications': ['when it started',
                            'where the symptom is located',
                            'how long it has been going on',
                            'how severe it is',
                            'associated symptoms'],
 'next_intent': 'ask_chest_pain',

In [1]:
import importlib

import intake_engine.policies.target_specs
import intake_engine.policies.base_complaint_policy
import intake_engine.policies.complaint_policies
import intake_engine.policies.policy_selector
import intake_engine.state.templates
import intake_engine.state.routing
import intake_engine.state.rule_based_parser
import intake_engine.llm_extractor
import intake_engine.IntakeState
import intake_engine.app_flow

importlib.reload(intake_engine.policies.target_specs)
importlib.reload(intake_engine.policies.base_complaint_policy)
importlib.reload(intake_engine.policies.complaint_policies)
importlib.reload(intake_engine.policies.policy_selector)
importlib.reload(intake_engine.state.templates)
importlib.reload(intake_engine.state.routing)
importlib.reload(intake_engine.state.rule_based_parser)
importlib.reload(intake_engine.llm_extractor)
importlib.reload(intake_engine.IntakeState)
importlib.reload(intake_engine.app_flow)

<module 'intake_engine.app_flow' from 'c:\\Source\\689\\Capstone2026_Spring\\intake_engine\\app_flow.py'>

In [2]:
from pprint import pprint
from intake_engine.app_flow import IntakeAppFlow

app = IntakeAppFlow(conn = None)

app.new_session(
    session_name = "headache skip regression test",
    primary_complaint = "headache",
    auto_save = False,
)

answers = [
    "I have a headache.",
    "No.",
    "No weakness, numbness, trouble speaking, or similar symptoms.",
    "No vision changes.",
    "No confusion.",
    "No fever or neck stiffness.",
    "No recent head trauma.",
    "No, I am not pregnant and not postpartum.",
    "This morning.",
    "About four hours.",
    "7 out of 10.",
    "It comes and goes.",
    "It has stayed about the same.",
    "In the front of my head.",
    "Throbbing.",
]

pprint(app.start_intake(auto_save = False))

for answer in answers:
    app.submit_answer(answer, auto_save = False)

result = app.submit_answer("Skip", auto_save = False)

print("AFTER 'Skip'")
pprint(result)
print()
print("QUESTION STATUS")
pprint(app.get_state()["conversation_meta"]["question_status"])
print()
print("HPI")
pprint(app.get_state()["hpi"])

{'intent': 'ask_main_reason_for_visit',
 'question': "Could you tell me what's been bothering you today?",
 'target': None}
AFTER 'Skip'
{'applied_update': {'append_fields': {},
                    'flags_to_add': [],
                    'missing_clarifications_to_add': [],
                    'set_fields': {'conversation_meta.last_answer_status': 'declined'}},
 'current_intent': 'ask_aggravating_factors',
 'missing_clarifications': ['associated symptoms'],
 'next_intent': 'ask_relieving_factors',
 'next_question': 'What makes the headache feel better?',
 'next_target': 'relieving_factors'}

QUESTION STATUS
{'aggravating_factors': 'asked_declined',
 'character': 'asked_answered',
 'confusion_or_ams': 'asked_answered',
 'course': 'asked_answered',
 'duration': 'asked_answered',
 'fever_or_neck_stiffness': 'asked_answered',
 'head_trauma': 'asked_answered',
 'location': 'asked_answered',
 'neurologic_symptoms': 'asked_answered',
 'onset': 'asked_answered',
 'pregnancy_or_postpartum_conte

In [3]:
from intake_engine.app_flow import IntakeAppFlow
from pprint import pprint

app = IntakeAppFlow(conn = None)

app.new_session(
    session_name = "ear pain debug",
    primary_complaint = None,
    auto_save = False,
)

pprint(app.start_intake(auto_save = False))

result = app.submit_answer("My ear hurts", auto_save = False)
pprint(result)

print()
print("CHIEF COMPLAINT")
print(app.get_state()["chief_complaint_primary"])

print()
print("LAST QUESTION INTENT")
print(app.get_state()["conversation_meta"]["last_question_intent"])

print()
print("LAST QUESTION TARGET")
print(app.get_state()["conversation_meta"]["last_question_target"])

{'intent': 'ask_main_reason_for_visit',
 'question': "Could you tell me what's been bothering you today?",
 'target': None}
{'applied_update': {'append_fields': {},
                    'flags_to_add': [],
                    'missing_clarifications_to_add': [],
                    'set_fields': {'chief_complaint_primary': 'ear pain',
                                   'conversation_meta.early_exit_reason': None,
                                   'conversation_meta.last_answer_status': 'answered'}},
 'current_intent': 'ask_main_reason_for_visit',
 'missing_clarifications': ['when it started',
                            'where the symptom is located',
                            'how long it has been going on',
                            'how severe it is',
                            'associated symptoms'],
 'next_intent': 'ask_fever_or_neck_stiffness',
 'next_question': 'Have you had a fever or a stiff neck with the headache?',
 'next_target': 'fever_or_neck_stiffness'}

CHIEF COMPL

In [4]:
from intake_engine.policies.policy_selector import get_policy_for_complaint

policy = get_policy_for_complaint("ear pain")

print(type(policy).__name__)
print(policy.policy_name)
print(policy.aliases)
print(policy.critical_followups)
print(policy.must_characterize)

BaseComplaintPolicy
ear_pain
['ear pain', 'ear ache', 'earache', 'my ear hurts', 'ear infection', 'ear drainage', "can't hear", 'hearing loss', 'ringing in ears', 'tinnitus', 'ear is ringing']
['fever_or_neck_stiffness', 'neurologic_symptoms', 'head_trauma', 'hearing_loss_or_tinnitus', 'ear_drainage_or_bleeding']
['onset', 'duration', 'severity', 'location', 'character', 'aggravating_factors', 'relieving_factors', 'associated_symptoms']
